# Unit Testing - Analisis Okupansi Ruangan
Notebook ini memuat setup awal untuk membaca data dari `dataset/data_.csv` dan beberapa cell di bawahnya berisi *unit test* terpisah untuk masing-masing fungsi dan class.

In [ ]:
import unittest
import os
from Models import SensorReading
from Repository import SensorRepository
from Analyzer import CO2Analyzer, ThermodynamicAnalyzer, EnergyEfficiencyAnalyzer

# Setup Global: Memuat data dari CSV asli agar bisa dipakai di seluruh cell testing
repo = SensorRepository()
file_path = 'dataset/data_.csv'

try:
    total_data = repo.load_csv(file_path)
    data_sensor = repo.get_all()
    print(f"[BERHASIL] {total_data} baris data dimuat dari {file_path}")
except FileNotFoundError:
    print(f"[GAGAL] File {file_path} tidak ditemukan. Pastikan path sesuai.")

### 1. Test Validasi Model (Models.py)

In [ ]:
class TestModelValidation(unittest.TestCase):
    def test_sensor_reading_validation(self):
        """Memastikan error ValueError muncul saat inisiasi data tidak valid"""
        # Test 1: Kelembapan (Humidity) lebih dari 100%
        with self.assertRaises(ValueError):
            SensorReading("2026-06-09", 25.0, 150.0, 100, 500.0, 0.004, 0)
        
        # Test 2: CO2 di bawah ambang batas (< 400)
        with self.assertRaises(ValueError):
            SensorReading("2026-06-09", 25.0, 50.0, 100, 300.0, 0.004, 0)

# Eksekutor spesifik hanya untuk cell ini
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestModelValidation)
unittest.TextTestRunner(verbosity=2).run(suite)

### 2. Test Fungsi is_occupied (Repository.py)

In [ ]:
class TestRepositoryLogic(unittest.TestCase):
    def test_get_occupied_filters_correctly(self):
        """Memastikan get_occupied() hanya mereturn data yang Occupancy-nya 1"""
        occupied_data = repo.get_occupied()
        
        # Cek apakah setiap baris hasil filter benar-benar occupied
        for row in occupied_data:
            self.assertTrue(row.is_occupied(), "Ada data yang terfilter tapi is_occupied() bernilai False")

# Eksekutor spesifik hanya untuk cell ini
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestRepositoryLogic)
unittest.TextTestRunner(verbosity=2).run(suite)

### 3. Test CO2 Analyzer (Analyzer.py)

In [ ]:
class TestCO2Analyzer(unittest.TestCase):
    def test_co2_analysis_structure(self):
        """Menguji output dari CO2Analyzer terhadap struktur Dictionary yang diharapkan"""
        analyzer = CO2Analyzer(data_sensor)
        result = analyzer.analyze()
        
        # Cek ketersediaan key pada dictionary return
        self.assertIn("overall_occupancy", result)
        self.assertIn("average_co2_when_occupied", result)
        self.assertIn("status", result)
        
        # Cek apakah status sesuai dengan opsi valid yang diset di kodingan
        self.assertIn(result["status"], ["Ventilasi Buruk", "Ventilasi Baik"])

# Eksekutor spesifik hanya untuk cell ini
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCO2Analyzer)
unittest.TextTestRunner(verbosity=2).run(suite)

### 4. Test Thermodynamic Analyzer (Analyzer.py)

In [ ]:
class TestThermodynamicAnalyzer(unittest.TestCase):
    def test_thermodynamic_calculations(self):
        """Menguji kalkulasi Max/Min Temperature dan rata-rata Humidity"""
        analyzer = ThermodynamicAnalyzer(data_sensor)
        result = analyzer.analyze()
        
        # Pastikan dictionary tidak kosong dan punya structure yang benar
        self.assertTrue(len(result) > 0)
        self.assertIn("temperature_range", result)
        
        # Max tidak boleh lebih kecil dari Min
        t_max = result["temperature_range"]["max"]
        t_min = result["temperature_range"]["min"]
        self.assertGreaterEqual(t_max, t_min, "Max temperature harus >= min temperature")

# Eksekutor spesifik hanya untuk cell ini
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestThermodynamicAnalyzer)
unittest.TextTestRunner(verbosity=2).run(suite)

### 5. Test Energy Efficiency Analyzer (Analyzer.py)

In [ ]:
class TestEnergyEfficiencyAnalyzer(unittest.TestCase):
    def test_energy_efficiency_waste_logic(self):
        """Menguji perhitungan pemborosan energi dan keakuratan matematis"""
        analyzer = EnergyEfficiencyAnalyzer(data_sensor)
        result = analyzer.analyze()
        
        waste_percentage = result.get("waste_percentage", 0)
        
        # Persentase tidak mungkin di bawah 0 atau di atas 100
        self.assertGreaterEqual(waste_percentage, 0.0)
        self.assertLessEqual(waste_percentage, 100.0)
        
        # Cek tipe data kembalian string untuk rekomendasi dan status
        self.assertIsInstance(result["status"], str)
        self.assertIsInstance(result["recommendation"], str)

# Eksekutor spesifik hanya untuk cell ini
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestEnergyEfficiencyAnalyzer)
unittest.TextTestRunner(verbosity=2).run(suite)